# Fine-Tune Whisper For Audio Classification with 🤗 Transformers

In [84]:
#***REMOVED***

from huggingface_hub import notebook_login
notebook_login()


You need account on Huggingface to source models and datasets as well as to uploud your results to the hub.

### Load Dataset
And reduce it only to two classes for testing.

In [85]:
from datasets import load_dataset, ClassLabel

dataset = load_dataset("speech_commands", "v0.02", verification_mode = "no_checks")

In [86]:
dataset = dataset.select_columns(["audio", "label"])
dataset['train'] = dataset['train'].select(range(300))
dataset['validation'] = dataset['validation'].select(range(300))
dataset['test'] = dataset['test'].select(range(300))

In [87]:
labels = dataset["train"].features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

id2label["0"]

'yes'

### Select your classes from the dataset

In [88]:
selected_classes = ["backward", "forward", "on", "off", "stop", "go", "zero", "one", "two", "three", "four", "five", "marvin", "sheila", "follow", "_silence_", "tree"]
# selected_classes = ["no", "go"]
dataset = dataset.filter(lambda example: id2label[str(example['label'])] in selected_classes)

In [89]:
print(dataset)

In [90]:
from transformers import AutoFeatureExtractor

model_id = "distil-whisper/distil-medium.en"
# model_id = "openai/whisper-tiny"
feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id, do_normalize=True
)

In [91]:
sampling_rate = feature_extractor.sampling_rate
sampling_rate

16000

In [92]:
from datasets import Audio

dataset = dataset.cast_column("audio", Audio(sampling_rate=sampling_rate))

### Preprocess Dataset

In [93]:
max_duration = 30

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=int(feature_extractor.sampling_rate * max_duration),
        truncation=True,
    )
    return inputs

In [94]:
dataset_encoded = dataset.map(
    preprocess_function,
    remove_columns="audio",
    batched=True,
    batch_size=2,
    num_proc=1,
)
dataset_encoded

DatasetDict({
    train: Dataset({
        features: ['label', 'input_features'],
        num_rows: 300
    })
    validation: Dataset({
        features: ['label', 'input_features'],
        num_rows: 153
    })
    test: Dataset({
        features: ['label', 'input_features'],
        num_rows: 9
    })
})

### Evaluate

In [95]:
import evaluate

accuracy = evaluate.load("accuracy")

In [96]:
import numpy as np


def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=eval_pred.label_ids)

In [100]:
from transformers import AutoModelForAudioClassification, TrainingArguments, Trainer

num_labels = len(id2label)
model_id = "openai/whisper-large"
model = AutoModelForAudioClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label, 
)


Some weights of WhisperForAudioClassification were not initialized from the model checkpoint at openai/whisper-large and are newly initialized: ['model.classifier.bias', 'model.classifier.weight', 'model.projector.bias', 'model.projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [101]:
from transformers import TrainingArguments

model_name = model_id.split("/")[-1]
batch_size = 2 # This affects your GPU VRAM requirements
gradient_accumulation_steps = 1
num_train_epochs = 10 # 10 seems the minimum for more than 5 classes

training_args = TrainingArguments(
    f"{model_name}-ft-kws-speech-commands",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_train_epochs,
    warmup_ratio=0.1,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
   
)

/home/niklas/PycharmProjects/Whisper/.venv/lib/python3.10/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  # eval_steps has to be defined and non-zero, fallbacks to logging_steps if the latter is non-zero


In [102]:
import evaluate
import numpy as np

metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [103]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=dataset_encoded["train"].with_format("torch"),
    eval_dataset=dataset_encoded["test"].with_format("torch"),
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics,
)

trainer.train()

/home/niklas/PycharmProjects/Whisper/.venv/lib/python3.10/site-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  if self.device.type in ["cpu", "xpu"]:


OutOfMemoryError: CUDA out of memory. Tried to allocate 30.00 MiB. GPU 0 has a total capacity of 7.79 GiB of which 17.62 MiB is free. Including non-PyTorch memory, this process has 7.76 GiB memory in use. Of the allocated memory 7.17 GiB is allocated by PyTorch, and 424.39 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
kwargs = {
    "dataset_tags": "speech_commands",
    "dataset": "Speech Commands",
    "model_name": f"{model_name}-ft-kws-speech-commands",
    "finetuned_from": model_id,
    "tasks": "audio-classification",
}

In [ ]:
trainer.push_to_hub(**kwargs)

### Inference (with local mic)

Or just load local file as "audio_file_path"

In [ ]:
import sounddevice as sd
import soundfile as sf

duration = 2
sr = 16000

rec = sd.rec(int(duration) * sr, samplerate=sr, channels=1)
sd.wait()

sd.play(rec, samplerate=sr)

audio_file_path = 'recorded_audio.wav'
sf.write(audio_file_path, rec, sr)


In [ ]:
from transformers import pipeline

classifier = pipeline("audio-classification", model="Teapack1/distilhubert-finetuned-no-go-kws")
classifier(audio_file_path)